In [1]:
import numpy as np
import pandas as pd

import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv("/content/drive/MyDrive/Student-Score-Predictor/data/processed/processed_data.csv")
df.head()

,Gender,EthnicGroup,ParentEduc,LunchType,TestPrep,ParentMaritalStatus,PracticeSport,IsFirstChild,NrSiblings,TransportMeans,WklyStudyHours,MathScore,ReadingScore,WritingScore
0,female,Unknown,bachelor's degree,standard,none,married,regularly,yes,3.0,school_bus,< 5,71,71,74
1,female,group C,some college,standard,Unknown,married,sometimes,yes,0.0,Unknown,5 - 10,69,90,88
2,female,group B,master's degree,standard,none,single,sometimes,yes,4.0,school_bus,< 5,87,93,91
3,male,group A,associate's degree,free/reduced,none,married,never,no,1.0,Unknown,5 - 10,45,56,42
4,male,group C,some college,standard,none,married,sometimes,yes,0.0,school_bus,5 - 10,76,78,75


In [3]:
df.columns

Index(['Gender', 'EthnicGroup', 'ParentEduc', 'LunchType', 'TestPrep',
       'ParentMaritalStatus', 'PracticeSport', 'IsFirstChild', 'NrSiblings',
       'TransportMeans', 'WklyStudyHours', 'MathScore', 'ReadingScore',
       'WritingScore'],
      dtype='object')

In [4]:
X_drop = ['MathScore','ReadingScore','WritingScore','NrSiblings']

In [5]:
X = df.drop(X_drop,axis=1)
y = df['MathScore']

In [6]:
X.columns

Index(['Gender', 'EthnicGroup', 'ParentEduc', 'LunchType', 'TestPrep',
       'ParentMaritalStatus', 'PracticeSport', 'IsFirstChild',
       'TransportMeans', 'WklyStudyHours'],
      dtype='object')

In [7]:
y.head()

,MathScore
0,71
1,69
2,87
3,45
4,76


In [8]:
ordinal_features = ['WklyStudyHours']
nominal_features = [
    'Gender',
    'EthnicGroup',
    'ParentEduc',
    'LunchType',
    'TestPrep',
    'ParentMaritalStatus',
    'PracticeSport',
    'IsFirstChild',
    'TransportMeans'
]

In [9]:
df['WklyStudyHours'].unique()

array(['< 5', '5 - 10', '> 10'], dtype=object)

In [10]:
study_hours_mapping = {
    '< 5': 0,
    '5 - 10': 1,
    '> 10': 2
}
X['WklyStudyHours'] = X['WklyStudyHours'].map(study_hours_mapping)

In [11]:
X['WklyStudyHours'].head()

,WklyStudyHours
0,0
1,1
2,0
3,1
4,1


In [12]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,y, test_size=0.2, random_state=42
)

In [14]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        (
          'nominal_encoder',
          OneHotEncoder(handle_unknown='ignore'),
          nominal_features
        )
    ],
    remainder='passthrough'
)

In [15]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [16]:
print(X_train_processed.shape)
print(X_test_processed.shape)

(23748, 35)
(5938, 35)


In [17]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [18]:
lr_model = LinearRegression()

lr_model.fit(X_train_processed,y_train)

LinearRegression()

In [19]:
y_pred = lr_model.predict(X_test_processed)

In [20]:
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("R2 Score:", r2)

MAE: 10.504897233514379
R2 Score: 0.27687386707366857


In [23]:
from sklearn.pipeline import Pipeline

In [25]:
pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('model', lr_model)
    ]
)
pipeline

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('nominal_encoder',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Gender', 'EthnicGroup',
                                                   'ParentEduc', 'LunchType',
                                                   'TestPrep',
                                                   'ParentMaritalStatus',
                                                   'PracticeSport',
                                                   'IsFirstChild',
                                                   'TransportMeans'])])),
                ('model', LinearRegression())])

In [26]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('nominal_encoder',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Gender', 'EthnicGroup',
                                                   'ParentEduc', 'LunchType',
                                                   'TestPrep',
                                                   'ParentMaritalStatus',
                                                   'PracticeSport',
                                                   'IsFirstChild',
                                                   'TransportMeans'])])),
                ('model', LinearRegression())])

In [27]:
y_pred_pipeline = pipeline.predict(X_test)

In [28]:
mae_pipeline = mean_absolute_error(y_test, y_pred_pipeline)
r2_pipeline = r2_score(y_test, y_pred_pipeline)

print("Pipeline MAE:", mae_pipeline)
print("Pipeline R2:", r2_pipeline)

Pipeline MAE: 10.504897233514379
Pipeline R2: 0.27687386707366857


In [29]:
from sklearn.ensemble import RandomForestRegressor

rf_pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('model', RandomForestRegressor(random_state=42))
    ]
)

In [31]:
rf_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('nominal_encoder',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Gender', 'EthnicGroup',
                                                   'ParentEduc', 'LunchType',
                                                   'TestPrep',
                                                   'ParentMaritalStatus',
                                                   'PracticeSport',
                                                   'IsFirstChild',
                                                   'TransportMeans'])])),
                ('model', RandomForestRegressor(random_state=42))])

In [32]:
rf_pred = rf_pipeline.predict(X_test)

In [33]:
rf_mae = mean_absolute_error(y_test, rf_pred)
rf_r2 = r2_score(y_test, rf_pred)

print("Random Forest MAE:", rf_mae)
print("Random Forest R2:", rf_r2)

Random Forest MAE: 11.717384405741923
Random Forest R2: 0.08717411280438636


In [34]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 5.5 MB/s eta 0:00:00


In [35]:
from catboost import CatBoostRegressor

In [36]:
cat_pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('model', CatBoostRegressor(
            random_state=42,
            verbose=0
        ))
    ]
)

In [37]:
cat_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('nominal_encoder',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Gender', 'EthnicGroup',
                                                   'ParentEduc', 'LunchType',
                                                   'TestPrep',
                                                   'ParentMaritalStatus',
                                                   'PracticeSport',
                                                   'IsFirstChild',
                                                   'TransportMeans'])])),
                ('model',
                 CatBoostRegressor(loss_function='RMSE', random_state=42, verbose=0))])

In [38]:
cat_pred = cat_pipeline.predict(X_test)

cat_mae = mean_absolute_error(y_test, cat_pred)
cat_r2 = r2_score(y_test, cat_pred)

print("CatBoost MAE:", cat_mae)
print("CatBoost R2:", cat_r2)

CatBoost MAE: 10.695875760128924
CatBoost R2: 0.24892648302999743


In [44]:
import joblib

joblib.dump(
    pipeline,
    f"/content/drive/MyDrive/Student-Score-Predictor/models/linear_regression_pipeline.pkl"
)

print("Pipeline saved successfully!")

Pipeline saved successfully!
